In [1]:
import os
import sys

sys.path.append("../../../../")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import copy
import torch
from datetime import datetime
from utils.helper import ModelConfig, color_print
from utils.dataset_utils.load_dataset import (
    load_data,
)
from utils.model_utils.save_module import save_module
from utils.model_utils.load_model import load_model
from utils.model_utils.evaluate import evaluate_model, get_sparsity, similar
from utils.dataset_utils.sampling import SamplingDataset
from utils.prune_utils.prune import prune_magnitude

In [3]:
name = "bert-tiny-yahoo"
device = torch.device("cuda:0")
checkpoint = None
batch_size = 16
num_workers = 4
num_samples = 128
magnitude_ratio = 0.3
seed = 44
include_layers = ["attention", "intermediate", "output"]
exclude_layers = None

In [4]:
script_start_time = datetime.now()
print(f"Script started at: {script_start_time.strftime('%Y-%m-%d %H:%M:%S')}")

Script started at: 2024-09-13 05:57:58


In [5]:
model_config = ModelConfig(name, device)
num_labels = model_config.config["num_labels"]
model, tokenizer, checkpoint = load_model(model_config)

Loading the model.

{'model_name': 'Models/bert-tiny-yahoo', 'tokenizer_name': 'sadickam/sdg-classification-bert', 'task_type': 'classification', 'architectures': 'bert', 'dataset_name': 'YahooAnswersTopics', 'num_labels': 10, 'cache_dir': 'Models'}

The model Models/bert-tiny-yahoo is loaded.

In [6]:
train_dataloader, valid_dataloader, test_dataloader = load_data(
    model_config.dataset_name,
    batch_size=batch_size,
    num_workers=num_workers,
    do_cache=True,
    seed=seed,
)

Loading cached dataset YahooAnswersTopics.

train.pkl is loaded from cache.

valid.pkl is loaded from cache.

test.pkl is loaded from cache.

The dataset YahooAnswersTopics is loaded

{'dataset_name': 'YahooAnswersTopics', 'path': 'yahoo_answers_topics', 'config_name': 'yahoo_answers_topics', 'text_column': 'question_title', 'label_column': 'topic', 'cache_dir': 'Datasets/YahooAnswersTopics', 'task_type': 'classification'}

In [7]:
all_samples = SamplingDataset(
    train_dataloader,
    200,
    num_samples,
    num_labels,
    False,
    4,
    device=device,
    resample=False,
    seed=seed,
)

In [8]:
# print("Evaluate the original model")
# result = evaluate_model(model, model_config, test_dataloader)

In [9]:
module = copy.deepcopy(model)
prune_magnitude(
    module,
    sparsity_ratio=magnitude_ratio,
    include_layers=include_layers,
    exclude_layers=exclude_layers,
)
print("Evaluate the pruned model")
result = evaluate_model(model, model_config, test_dataloader)
# save_module(module, "Modules/", f"magnitude_{name}_{magnitude_ratio}p.pt")

Evaluate the pruned model

Evaluating the model:   0%|                                                   | 0/1875 [00:00<?, ?it/s]

Loss: 1.2292

Precision: 0.6310, Recall: 0.6152, F1-Score: 0.6147

              precision    recall  f1-score   support

           0       0.55      0.47      0.50      2941
           1       0.69      0.53      0.60      2997
           2       0.65      0.67      0.66      3016
           3       0.37      0.58      0.45      2978
           4       0.68      0.80      0.74      3017
           5       0.72      0.81      0.76      3004
           6       0.66      0.39      0.49      3037
           7       0.60      0.61      0.61      3026
           8       0.65      0.67      0.66      2997
           9       0.75      0.63      0.69      2987

    accuracy                           0.62     30000
   macro avg       0.63      0.62      0.61     30000
weighted avg       0.63      0.62      0.61     30000


In [10]:
for concern in range(num_labels):
    print(f"--{concern}--")
    valid = copy.deepcopy(valid_dataloader)
    similar(
        model, module, valid, concern, num_samples, num_labels, device=device, seed=seed
    )

--0--

adding eps to diagonal and taking inverse

taking square root

dot products...

trying to take final svd

...

CCA coefficients mean non-concern: (0.9996222432248112, 0.9996222432248112)

Linear CKA concern: 0.998495936283884

Linear CKA non-concern: 0.9988784829809355

Kernel CKA concern: 0.9984375307866069

Kernel CKA non-concern: 0.9986852500664308

In [11]:
get_sparsity(module)

(0.2853949221188323,
 {'bert.encoder.layer.0.attention.self.query.weight': 0.29998779296875,
  'bert.encoder.layer.0.attention.self.query.bias': 0.0,
  'bert.encoder.layer.0.attention.self.key.weight': 0.29998779296875,
  'bert.encoder.layer.0.attention.self.key.bias': 0.0,
  'bert.encoder.layer.0.attention.self.value.weight': 0.29998779296875,
  'bert.encoder.layer.0.attention.self.value.bias': 0.0,
  'bert.encoder.layer.0.attention.output.dense.weight': 0.29998779296875,
  'bert.encoder.layer.0.attention.output.dense.bias': 0.0,
  'bert.encoder.layer.0.intermediate.dense.weight': 0.29998779296875,
  'bert.encoder.layer.0.intermediate.dense.bias': 0.0,
  'bert.encoder.layer.0.output.dense.weight': 0.29998779296875,
  'bert.encoder.layer.0.output.dense.bias': 0.0,
  'bert.encoder.layer.1.attention.self.query.weight': 0.29998779296875,
  'bert.encoder.layer.1.attention.self.query.bias': 0.0,
  'bert.encoder.layer.1.attention.self.key.weight': 0.29998779296875,
  'bert.encoder.layer.1.at